# Minimal Translation Transformer

In [16]:
import torch
import torch.nn as nn
import math

## Positional Encoding

In [ ]:
# Sinusoidal positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

# Learnable positional encoding
class PositionalEncoding(nn.Module):

    def __init__(self, max_len, d_model):
        super().__init__()

        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):

        batch_size, seq_len, d_model = x.size()

        positions = torch.arange(seq_len, device=x.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        pos_embed = self.pos_embedding(positions)

        return x + pos_embed

## Encoder

In [4]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, ff_dim, n_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead = n_heads,
            dim_feedforward=ff_dim,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

    def forward(self, src):
        x = self.embedding(src)
        x = self.pos(x)
        return self.encoder(x)

## Decoder

In [5]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, ff_dim, n_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(layer, num_layers=n_layers)

    def forward(self, tgt, memory, tgt_mask=None):
        x = self.embedding(tgt)
        x = self.pos(x)

        return self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask
        )

## Full Transformer

In [11]:
class TranslationTransformer(nn.Module):
    def __init__(
        self,
        src_vocab,
        tgt_vocab,
        d_model=256,
        n_heads=4,
        ff_dim=512,
        n_layers=2
    ):
        super().__init__()

        self.encoder = Encoder(src_vocab, d_model, n_heads, ff_dim, n_layers)
        self.decoder = Decoder(tgt_vocab, d_model, n_heads, ff_dim, n_layers)

        self.fc = nn.Linear(d_model, tgt_vocab)

    def generate_mask(self, size):
        mask = torch.triu(torch.ones(size, size), diagonal=1)
        mask = mask.masked_fill(mask == 1, float("-inf"))
        return mask

    def forward(self, src, tgt):

        memory = self.encoder(src)

        tgt_mask = self.generate_mask(tgt.size(1)).to(tgt.device)

        out = self.decoder(
            tgt,
            memory,
            tgt_mask=tgt_mask
        )

        return self.fc(out)

In [14]:
#inference

src_vocab = 8000
tgt_vocab = 8000

model = TranslationTransformer(src_vocab, tgt_vocab)

src = torch.randint(0, src_vocab, (32, 20))
tgt = torch.randint(0, tgt_vocab, (32, 20))

out = model(src, tgt)

print(out.shape)

torch.Size([32, 20, 8000])
